# Демонстрация работы ZIP-архиватора через командную строку
Проверка основных сценариев: справка, упаковка файлов (`pack`), просмотр содержимого (`list`) и распаковка (`unpack`).

In [1]:
import subprocess
import shutil
from pathlib import Path

# 1. Автоматически ищем собранный exe-файл проекта
exe_files = list(Path('.').rglob('Archiver.WinForms.exe'))
assert exe_files, 'Сначала соберите проект в Visual Studio (Ctrl+Shift+B)!'
exe_path = str(exe_files[0])
print('Найден исполняемый файл:', exe_path)

# 2. Готовим тестовую папку с файлами
work_dir = Path('demo_workspace')
if work_dir.exists():
    shutil.rmtree(work_dir)

src_dir = work_dir / 'source_files'
src_dir.mkdir(parents=True)
(src_dir / 'readme.txt').write_text('Тестовый текстовый файл для проверки архиватора. ' * 20, encoding='utf-8')
(src_dir / 'notes.txt').write_text('Заметки по предмету Основы разработки мобильных приложений.', encoding='utf-8')
print('Созданы тестовые файлы в папке:', src_dir)

Найден исполняемый файл: Archiver.WinForms\bin\Debug\net10.0-windows\Archiver.WinForms.exe
Созданы тестовые файлы в папке: demo_workspace\source_files


In [2]:
# 3. Вызов справки по параметрам командной строки (--help)
res = subprocess.run([exe_path, '--help'], capture_output=True, text=True, encoding='cp866', errors='replace')
print(res.stdout)

=== Справка по параметрам командной строки ===
  Archiver.WinForms.exe                                     - Запуск графического интерфейса
  Archiver.WinForms.exe <файл.zip>                          - Открыть архив в графическом окне
  Archiver.WinForms.exe pack <путь> <архив.zip> [режим]     - Упаковать файл/папку (режимы: optimal, fastest, none)
  Archiver.WinForms.exe unpack <архив.zip> <папка>          - Распаковать архив
  Archiver.WinForms.exe list <архив.zip>                    - Вывести список файлов внутри архива



In [3]:
# 4. Упаковка папки в ZIP-архив (команда pack)
archive_path = str(work_dir / 'result.zip')
res = subprocess.run([exe_path, 'pack', str(src_dir), archive_path, 'optimal'], capture_output=True, text=True, encoding='cp866', errors='replace')
print(res.stdout)

[ 50%] source_files/notes.txt
[100%] source_files/readme.txt
Готово! Архив создан: d:\Projects\ArchiverSolution\demo_workspace\result.zip



In [4]:
# 5. Просмотр содержимого архива без распаковки (команда list)
res = subprocess.run([exe_path, 'list', archive_path], capture_output=True, text=True, encoding='cp866', errors='replace')
print(res.stdout)

Содержимое архива result.zip (файлов: 2):
 - source_files/notes.txt              | Было:      111 Б | Стало:       82 Б | Сжатие: 26%
 - source_files/readme.txt             | Было:     1,8 КБ | Стало:       92 Б | Сжатие: 95%



In [5]:
# 6. Распаковка архива в новую папку (команда unpack)
extract_dir = work_dir / 'extracted'
res = subprocess.run([exe_path, 'unpack', archive_path, str(extract_dir)], capture_output=True, text=True, encoding='cp866', errors='replace')
print(res.stdout)

print('Распакованные файлы:')
for f in extract_dir.rglob('*'):
    if f.is_file():
        print(' ->', f)

[ 50%] source_files/notes.txt
[100%] source_files/readme.txt
Готово! Файлы распакованы в: d:\Projects\ArchiverSolution\demo_workspace\extracted

Распакованные файлы:
 -> demo_workspace\extracted\source_files\notes.txt
 -> demo_workspace\extracted\source_files\readme.txt
